# TP2 — Generation : des passages pertinents à la réponse

**Objectif de l'après-midi** : ajouter le **G** de RAG. Les k chunks trouvés au TP1 sont donnés à un LLM,
avec la question, pour qu'il rédige une réponse **appuyée sur les documents et vérifiable**.

```
question ──► search (TP1) ──► k chunks ──┐
                                          ├──► prompt ──► LLM ──► réponse + sources
consignes (system) + question ────────────┘
```

Plan :
1. Appeler un LLM via une API
2. Sans RAG : le LLM seul
3. Avec RAG
4. Expériences
5. Exercices

Prérequis : l'index du TP1 (`index/`) et la clé d'API dans `.env` (voir `README.md`).

In [1]:
import time

from dotenv import load_dotenv
from litellm import completion

import settings
from rag import load_index, search, build_prompt, generate, answer

load_dotenv()  # charge la clé d'API (LLM_API_KEY) depuis .env
chunks, embeddings = load_index()  # l'index construit au TP1
print(len(chunks), "chunks,", embeddings.shape[1], "dimensions")

601 chunks, 384 dimensions


## 1. Appeler un LLM via une API

Un LLM de plusieurs milliards de paramètres tourne sur des GPU distants : on l'appelle via une **API HTTP**.
Tous les fournisseurs (OpenAI, Mistral, Anthropic, Hugging Face, …) ont adopté le même format de conversation :
une liste de **messages**, chacun avec un rôle :

- `system` : les consignes données au modèle (ton, format, limites) ;
- `user` : ce que dit l'utilisateur ;
- `assistant` : ce que répond le modèle.

La librairie **LiteLLM** expose une seule fonction, `completion()`, quel que soit le fournisseur.
Ici on passe par le **proxy LLM de Cross Data** (une API compatible OpenAI, adresse dans `settings.LLM_API_BASE`) :
le modèle est choisi dans `settings.GENERATION_MODEL`, la clé d'API est lue dans `.env`.

In [ ]:
messages = [
    {"role": "system", "content": "Tu es un assistant concis. Réponds en deux phrases maximum."},
    {"role": "user", "content": "Qu'est-ce qu'un capteur à ultrasons ?"},
]

reponse = completion(model=settings.GENERATION_MODEL, messages=messages, temperature=0)

print(reponse.choices[0].message.content)
print("\nusage :", reponse.usage)

`usage` donne le nombre de tokens **envoyés** (`prompt_tokens`) et **générés** (`completion_tokens`) : c'est ce qui est facturé,
et c'est ce qui doit tenir dans la fenêtre de contexte du modèle.

La **température** règle le hasard dans le choix de chaque token : 0 = le token le plus probable à chaque fois ; 2 = plus de variété.

In [ ]:
messages = [{"role": "user", "content": "Propose un nom original pour un robot explorateur. Réponds par le nom seul."}]

for temperature in (0.0, 0.0, 1.5, 1.5):
    reponse = completion(model=settings.GENERATION_MODEL, messages=messages, temperature=temperature, max_tokens=20)
    print(f"température {temperature} → {reponse.choices[0].message.content.strip()}")

## 2. Sans RAG : le LLM seul

Posons une question précise sur nos datasheets, **sans lui donner les documents**.

In [ ]:
question = "Quelle est la consommation de courant du BME280 en mode veille (sleep mode) ?"

reponse = completion(model=settings.GENERATION_MODEL, messages=[{"role": "user", "content": question}], temperature=0)
print(reponse.choices[0].message.content)

**À observer**

Le modèle répond avec assurance… mais a-t-il raison ? Il n'a que ce qu'il a mémorisé pendant son entraînement
(peut-être cette datasheet, peut-être pas, peut-être une version différente). Rien ne permet de le vérifier :
c'est le problème des **hallucinations**. Vérifiez dans `data/bme280.pdf` (chapitre "Electrical characteristics").

## 3. Avec RAG

Trois étapes :
1. **chercher** les chunks pertinents (TP1 : `search`) ;
2. **construire le prompt** : consignes + extraits + question (`build_prompt`) ;
3. **générer** la réponse (`generate`).

Regardons d'abord le prompt réellement envoyé au modèle.

In [ ]:
extraits = search(question, chunks, embeddings, k=settings.TOP_K)   # 1. retrieval
messages = build_prompt(question, extraits)                         # 2. prompt

print("=== system ===")
print(messages[0]["content"])
print("\n=== user (début) ===")
print(messages[1]["content"][:1500], "\n[...]")

In [ ]:
print(generate(messages))                                           # 3. generation

**À observer**

- La connaissance n'est plus dans le modèle : elle est **dans le prompt**.
- La réponse cite `[fichier, page]`. On peut donc **vérifier** les réponses. C'est le principal intérêt du RAG en entreprise.
- Le prompt système impose des règles : répondre uniquement à partir des extraits, dire quand l'information n'y est pas, citer.

`answer` dans `rag.py` enchaîne les trois étapes et renvoie la réponse **et** les chunks utilisés.

In [ ]:
questions = [
    "Quelle est la plage de tension d'alimentation (VSYS) du Raspberry Pi Pico ?",
    "Quelle est la fréquence maximale du processeur de l'ESP32 ?",
    "Quelle est la précision du LM35 à 25 °C ?",
]

for q in questions:
    reponse, utilises = answer(q, chunks, embeddings)
    print("###", q)
    print(reponse)
    print("chunks utilisés :", ", ".join(f"{c['source']} p.{c['page']}" for c in utilises))
    print()

Et si la réponse n'est **pas** dans les documents ? Un bon RAG doit le dire plutôt qu'inventer.

In [ ]:
reponse, utilises = answer("Quel est le prix du Raspberry Pi Pico ?", chunks, embeddings)
print(reponse)

## 4. Expériences

### 4.a Combien de chunks donner au modèle ?

`k` (dans `settings.TOP_K`) est un compromis : plus de chunks = plus de chances que la réponse y soit,
mais plus de tokens (coût, latence), plus de bruit pour le modèle, et une fenêtre de contexte plus vite atteinte.

In [ ]:
question = "Quelle est la consommation de courant du BME280 en mode veille (sleep mode) ?"

for k in (1, 5, 15):
    messages = build_prompt(question, search(question, chunks, embeddings, k=k))
    reponse = completion(model=settings.GENERATION_MODEL, messages=messages, temperature=0, max_tokens=200)
    print(f"k={k:<3} tokens envoyés : {reponse.usage.prompt_tokens:<6} → {reponse.choices[0].message.content[:160]!r}")

### 4.b Petit modèle + RAG ou gros modèle seul ?

Comparons le modèle par défaut avec un modèle plus gros (`settings.GENERATION_MODEL_LARGE`), avec et sans RAG,
sur la même question.

*Le proxy n'expose qu'un seul modèle pour l'instant : « gros » et « petit » sont identiques et les résultats aussi.
L'expérience prendra son sens dès qu'un modèle plus gros sera ajouté dans `settings.py`.*

In [ ]:
essais = [
    ("petit modèle, sans RAG", settings.GENERATION_MODEL, False),
    ("petit modèle + RAG", settings.GENERATION_MODEL, True),
    ("gros modèle, sans RAG", settings.GENERATION_MODEL_LARGE, False),
    ("gros modèle + RAG", settings.GENERATION_MODEL_LARGE, True),
]

for nom, modele, avec_rag in essais:
    if avec_rag:
        messages = build_prompt(question, search(question, chunks, embeddings))
    else:
        messages = [{"role": "user", "content": question}]
    debut = time.time()
    reponse = completion(model=modele, messages=messages, temperature=0, max_tokens=200)
    duree = time.time() - debut
    print(f"### {nom} — {duree:.1f} s, {reponse.usage.total_tokens} tokens")
    print(reponse.choices[0].message.content.strip(), "\n")

**À observer**

Un petit modèle **bien alimenté en contexte** fait souvent aussi bien qu'un gros modèle livré à sa mémoire —
pour une fraction du coût, de la latence et de l'énergie. C'est l'argument de **frugalité** du RAG :
la connaissance est dans les documents (gratuits à lire), l'intelligence du modèle ne sert qu'à la restituer.

## 5. Exercices

Tout se fait dans les cellules ci-dessous ; `rag.py` reste inchangé.

### Exercice 1 — le prompt système

La cellule est complète et s'exécute telle quelle : l'exercice consiste à **modifier le texte de `SYSTEM_PROMPT`**
(format de réponse — liste, tableau —, longueur, unités, comportement quand la réponse est absente, position des
citations…) et à observer l'effet sur les réponses.

In [ ]:
SYSTEM_PROMPT = """Tu es un assistant technique qui répond à des questions sur des datasheets de composants électroniques.
Réponds UNIQUEMENT à partir des extraits fournis. Si l'information n'y figure pas, dis-le clairement.
Cite tes sources à la fin de chaque affirmation sous la forme [fichier, page N].
Réponds dans la langue de la question."""  # <- à modifier, puis ré-exécuter la cellule


def answer_avec_consignes(question, chunks, embeddings, system_prompt, k=settings.TOP_K):
    # Comme rag.answer, mais le prompt système est passé en paramètre
    extraits = search(question, chunks, embeddings, k)
    messages = build_prompt(question, extraits)
    messages[0] = {"role": "system", "content": system_prompt}
    return generate(messages), extraits


for q in ["Quelle est la précision du LM35 à 25 °C ?", "Quel est le prix du Raspberry Pi Pico ?"]:
    reponse, _ = answer_avec_consignes(q, chunks, embeddings, SYSTEM_PROMPT)
    print("###", q)
    print(reponse)
    print()


### Exercice 2 — la recherche hybride dans le RAG

Collez dans la cellule vos fonctions du TP1, puis décommentez le test : les questions sur des identifiants exacts
(`ctrl_meas`…) s'améliorent-elles ?

In [ ]:
import re

import numpy as np
from rank_bm25 import BM25Okapi

# Collez ici vos fonctions du TP1 : MOTS_VIDES, tokenize_sans_mots_vides,
# search_bm25_sans_mots_vides et search_hybrid.


def answer_hybride(question, chunks, embeddings, k=settings.TOP_K):
    # Comme rag.answer, mais le retrieval est hybride au lieu de purement sémantique
    extraits = search_hybrid(question, chunks, embeddings, k)
    return generate(build_prompt(question, extraits)), extraits


# à décommenter une fois search_hybrid en place :
# for q in ["À quelle adresse se trouve le registre ctrl_meas du BME280 ?",
#           "Quelle est la tension d'alimentation du HC-SR04 ?"]:
#     reponse, utilises = answer_hybride(q, chunks, embeddings)
#     print("###", q)
#     print(reponse)
#     print("chunks utilisés :", ", ".join(f"{c['source']} p.{c['page']}" for c in utilises))
#     print()


### Exercice 3 — réécrire la question avant la recherche

Les datasheets sont en anglais, les questions en français. Complétez `rewrite_query` : demander au LLM une requête de
recherche courte, en anglais technique, avec les termes exacts qu'on trouverait dans une datasheet. Le LLM sert ici
*aussi* à améliorer le retrieval. Comparez ensuite les chunks trouvés avec et sans réécriture.

In [ ]:
def rewrite_query(question):
    # À compléter : appeler le LLM avec des consignes de réécriture,
    # et renvoyer uniquement la requête produite.
    messages = [
        {"role": "system", "content": ...},
        {"role": "user", "content": question},
    ]
    ...


# à décommenter une fois la fonction complétée :
# question = "Quelle est la tension d'alimentation du HC-SR04 ?"
# requete = rewrite_query(question)
# print("question :", question, "\nrequête  :", requete)
#
# print("\n--- recherche avec la question ---")
# for c in search(question, chunks, embeddings, k=5):
#     print(f"{c['score']:.3f}  {c['source']:<16} p.{c['page']}")
# print("\n--- recherche avec la requête réécrite ---")
# for c in search(requete, chunks, embeddings, k=5):
#     print(f"{c['score']:.3f}  {c['source']:<16} p.{c['page']}")


### Bonus — un RAG conversationnel

Pour que « et l'ESP32 ? » ait un sens après une question sur le Pico, il faut renvoyer au modèle **toute la discussion**
à chaque tour. Complétez `chat`. Question à se poser : que faire de l'historique quand il devient trop long pour la
fenêtre de contexte ?

In [ ]:
historique = []  # la discussion complète, renvoyée au modèle à chaque tour


def chat(message, k=settings.TOP_K):
    # À compléter :
    # 1. chercher les extraits pertinents pour le nouveau message
    # 2. ajouter à `historique` le message utilisateur, accompagné des extraits (voir build_prompt)
    # 3. appeler le LLM avec tout l'historique, ajouter sa réponse à l'historique, l'afficher
    ...


# chat("Combien de broches GPIO possède le Raspberry Pi Pico ?")
# chat("et l'ESP32 ?")   # cette question n'a de sens que grâce à l'historique


## Bilan

```
PDF ──► chunks ──► embeddings ──► index                     (indexation, TP1)
question ──► search ──► k chunks ──► prompt ──► LLM ──► réponse + sources   (TP1 + TP2)
```

Toutes les briques ont des réglages (découpage, modèle d'embeddings, k, hybride, prompt, modèle de génération…).
Pour choisir sans se fier à l'impression laissée par trois questions, il faut **mesurer** : c'est le TP3.